# Deconvolve Positive Control Genes and plot
February 8, 2024

This notebook is designed to simplify the deconvolution process for gene expression and chromatin in one pass. The functions save the output to disk as plots, and eventually as data and metadata for downstream analysis.


In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
def deconvolve_gene(gene_name, replicate, save_dir):
    """
    Deconvolve a gene's gene expression and chromatin
    """
    from src.model import Model
    from cc_src.chromatin_model import ChromatinModel
    from src.config import load_yl_replicate1_rg1_alpha_vst_config, \
        load_yl_replicate2_rg1_alpha_vst_config
    
    if replicate == 1:
        config = load_yl_replicate1_rg1_alpha_vst_config()
    elif replicate == 2:
        config = load_yl_replicate2_rg1_alpha_vst_config()
    else:
        raise ValueError("Undefined replicate")
    
    print(f"Deconvolving {gene_name}, replicate={replicate}")
    print(f"Deconvolving gene expression...", end="")
    ge_model = Model(config, gene_name)
    ge_model.deconvolve_find_optimal_gamma()
    print("Done.")

    print(f"Deconvolving chromatin...", end="")
    chrom_model = ChromatinModel(config)
    chrom_model.load_mnase_gene(gene_name, replicate=config.replicate)
    chrom_model.deconvolve_find_optimal_gamma()
    print("Done.")
    
    save_name = f"{save_dir}/{gene_name}_rep{config.replicate}_chromatin.png"
    fig = chrom_model.create_deconvolution_plots_abbreviated_flipped(ge_model=ge_model)
    plt.savefig(save_name, dpi=200)
    print(f"Saved figure to {save_name}")
    plt.close(fig)

    save_name = f"{save_dir}/{gene_name}_{config.replicate}_predicted.png"
    fig = chrom_model.plot_prediction_comparison()
    plt.savefig(save_name, dpi=200)
    print(f"Saved figure to {save_name}")
    plt.close(fig)


In [ ]:
from cc_src.geneset import positive_control_genes
from src.timer import Timer

save_dir1 = "output/positive_controls_chromatin/rep1"
save_dir2 = "output/positive_controls_chromatin/rep2"

save_dirs = {
    1: save_dir1,
    2: save_dir2
}

# For each replicate, deconvolve the positive control genes
# and save the resulting figures to disk
# TODO: Save the deconvolved data and metadata to disk
#       Can use existing code for this, but requires some cleanup
#       and simplification
timer = Timer()

for replicate in [1, 2]:
    save_dir = save_dirs[replicate]
    for gene_name in positive_control_genes():
        deconvolve_gene(gene_name, replicate, save_dir)
        print()
        timer.get_time()


Deconvolving CLB2, replicate=1
Deconvolving gene expression...Done.
Deconvolving chromatin...Loading MNase reads for CLB2...Done.
The histogram shape around the TSS is: (3, 9)
The shape of the flattened grid to be deconvolved is: (16, 27)
Applying normalization using scaling matrix: output/mnase/rep1_len_scaling_3len_bins.csv
The shape of the flattened grid to be deconvolved is: (16, 27)
Running the find optimal gamma procedure...
  ... The base fitting norm (rn) with no smoothing (gamma=0) is: 0.8319
  ... Searching for an optimal gamma value in the boundaries: [0.0001, 0.0100]
  ...  search left, rn_goal = 0.9119, rate = 9.6
  ...   gm = 0.0050, rn = 1.0323, rate = 24.09, time = 00:00:35.47
  ...   gm = 0.0026, rn = 0.9649, rate = 15.99, time = 00:00:44.25
  ...   gm = 0.0013, rn = 0.9233, rate = 10.99, time = 00:00:53.34
  ...  search right, rn_goal = 1.1647, rate = 40.0
  ... rn range: [0.9233, 1.1355]
  ... search gamma in [0.0013 0.0100] for elbow
  ...   gm = 0.0013, rn = 0.9233